# Week 7 Assignment: Document Question Answering System using RAG

This notebook is my project for the RAG assignment. The idea is to build a system that can read a document, break it into pieces, store those pieces in a way that can be searched, and then use that to answer questions about the document.

I used FAISS for storing the vectors, BM25 for keyword search, and Qwen2.5-1.5B-Instruct as the language model that generates the final answer.

I ran everything on Google Colab using a T4 GPU since the language model needs some GPU memory to run smoothly.


In [1]:
!pip install -qU pypdf sentence-transformers faiss-cpu rank-bm25 transformers accelerate datasets ipywidgets
print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.0 MB/s eta 0:00:00
Installation complete.


In [2]:
import re, time, gc, torch, faiss
import numpy as np
import pandas as pd
import ipywidgets as widgets

from pypdf import PdfReader
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, clear_output

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 1. Document Ingestion

First step is just getting the document into the notebook. I upload a PDF and pull the raw text out of it page by page. I also added a text file loader in case someone wants to test with a plain .txt file instead.


In [3]:
from google.colab import files

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

if not uploaded_files:
    raise ValueError("No file uploaded.")

DOC_PATH = uploaded_files[0]
print("Selected file:", DOC_PATH)

Saving Himanshu_Singh_Gahlot_Resume.pdf to Himanshu_Singh_Gahlot_Resume.pdf
Selected file: Himanshu_Singh_Gahlot_Resume.pdf


In [4]:
def load_pdf(file_path):
    reader = PdfReader(file_path)
    documents = []
    for page_number, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            documents.append({
                "text": text,
                "source": file_path,
                "page": page_number + 1
            })
    return documents

def load_text_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read()
    return [{"text": text, "source": file_path, "page": None}]

def load_huggingface_data(dataset_name, split="train", text_column="text", max_samples=100):
    dataset = load_dataset(dataset_name, split=split)
    documents = []
    for item in dataset.select(range(min(max_samples, len(dataset)))):
        text = str(item[text_column])
        if text.strip():
            documents.append({"text": text, "source": dataset_name, "page": None})
    return documents

if DOC_PATH.lower().endswith(".pdf"):
    raw_documents = load_pdf(DOC_PATH)
elif DOC_PATH.lower().endswith(".txt"):
    raw_documents = load_text_file(DOC_PATH)
else:
    raise ValueError("Upload a PDF or TXT file.")

print("Sections/pages loaded:", len(raw_documents))

Sections/pages loaded: 1


In [5]:
for i, doc in enumerate(raw_documents[:3], start=1):
    print(f"\n--- Section {i} ---")
    print("Page:", doc["page"])
    print(doc["text"][:1000])


--- Section 1 ---
Page: 1
Himanshu Singh Gahlot
Jaipur
♂phone+91-8890795885 |/envel⌢pehimanshusinghgahlot1312@gmail.com |/linkedin-inLinkedIn |/githubGitHub |/gl⌢bePortfolio
SUMMARY
Final-year Computer Science student with a focused interest in Data Science and Machine Learning. Have
worked with real datasets, built and evaluated ML models, and deployed them as usable applications. Com-
fortable in Python and Linux, picks up new tools quickly, and prefers working on problems where results can
be measured and iterated on.
EDUCATION
Bachelor of Technology in Computer Science and Engineering2023 – 2027
Jaipur Engineering College and Research Centre — CGPA: 8.83 (till 5th Sem) Jaipur, Rajasthan
Senior Secondary Education — 83.40%2022
Vikas Adarsh Model Sr. Sec. School Bikaner, Rajasthan
Secondary Education — 88.17%2020
Govt. Sadul Sr. Sec. School Bikaner, Rajasthan
SKILLS
Programming Languages:C++, Python
AI / ML:Python (NumPy, Pandas, NLTK), ML Algorithms, Data Preprocessing
Computer Fun

## 2. Text Cleaning and Chunking

The text coming out of a PDF is messy, so I clean it up a bit first (removing extra line breaks and spaces). After that I split it into chunks. I wasn't sure what chunk size would work best so I made three versions, small, medium and large, and compared them later.


In [6]:
def clean_text(text):
    text = text.replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()

cleaned_documents = [
    {"text": clean_text(d["text"]), "source": d["source"], "page": d["page"]}
    for d in raw_documents
]

def create_chunks(documents, chunk_size=700, chunk_overlap=100):
    chunks = []
    for doc_id, doc in enumerate(documents):
        text, start, chunk_id = doc["text"], 0, 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunk_text = text[start:end]

            if end < len(text):
                last_break = max(
                    chunk_text.rfind(". "),
                    chunk_text.rfind("? "),
                    chunk_text.rfind("! ")
                )
                if last_break > chunk_size * 0.6:
                    end = start + last_break + 1
                    chunk_text = text[start:end]

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text.strip(),
                    "source": doc["source"],
                    "page": doc["page"],
                    "document_id": doc_id,
                    "chunk_id": chunk_id
                })
                chunk_id += 1

            if end >= len(text):
                break
            start = max(end - chunk_overlap, start + 1)
    return chunks

chunk_strategies = {
    "small_overlap": {"chunk_size": 200, "chunk_overlap": 30},
    "medium_overlap": {"chunk_size": 400, "chunk_overlap": 60},
    "large_overlap": {"chunk_size": 700, "chunk_overlap": 100}
}

chunk_results = {}
rows = []

for name, config in chunk_strategies.items():
    chunks = create_chunks(cleaned_documents, **config)
    chunk_results[name] = chunks
    rows.append({
        "Strategy": name,
        "Number of Chunks": len(chunks),
        "Average Length": round(np.mean([len(c["text"]) for c in chunks]), 2)
    })

chunk_comparison_df = pd.DataFrame(rows)
chunk_comparison_df

,Strategy,Number of Chunks,Average Length
0,small_overlap,18,189.83
1,medium_overlap,10,345.00
2,large_overlap,6,568.33


In [7]:
CHOSEN_STRATEGY = "large_overlap"
chosen_chunks = chunk_results[CHOSEN_STRATEGY]
chunk_texts = [chunk["text"] for chunk in chosen_chunks]

print("Selected strategy:", CHOSEN_STRATEGY)
print("Total chunks:", len(chosen_chunks))

Selected strategy: large_overlap
Total chunks: 6


## 3. Embeddings and FAISS Vector Database

Now each chunk gets turned into a vector using a sentence embedding model. I store all these vectors in FAISS so I can quickly search through them later instead of comparing the question to every chunk manually.


In [8]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

start = time.time()
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

vector_store = faiss.IndexFlatIP(chunk_embeddings.shape[1])
vector_store.add(chunk_embeddings)

print("Embedding shape:", chunk_embeddings.shape)
print("Vectors stored:", vector_store.ntotal)
print(f"Embedding time: {time.time() - start:.2f}s")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (6, 384)
Vectors stored: 6
Embedding time: 0.99s


## 4. Query Processing and Dense Retrieval

When someone asks a question, it needs to be converted into a vector the same way the chunks were, using the same embedding model. Then I search the FAISS index to find the chunks that are closest to the question.


In [9]:
def create_query_embedding(query, model=embedding_model):
    return model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

def dense_search(query, top_k=6):
    query_vector = create_query_embedding(query)
    scores, indices = vector_store.search(query_vector, top_k)
    results = []

    for score, index in zip(scores[0], indices[0]):
        if index != -1:
            results.append({
                "index": int(index),
                "text": chosen_chunks[index]["text"],
                "score": float(score),
                "source": chosen_chunks[index]["source"],
                "page": chosen_chunks[index]["page"]
            })
    return results

sample_query = "What is the main idea of this document?"
print("Query vector shape:", create_query_embedding(sample_query).shape)

Query vector shape: (1, 384)


## 5. BM25 Keyword Retrieval

Vector search is good at understanding meaning but sometimes it misses exact words, like a specific score or a certification name. So I also built a BM25 keyword search as a second way of finding relevant chunks.


In [10]:
def tokenize_for_bm25(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_chunks = [tokenize_for_bm25(text) for text in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

def bm25_search(query, top_k=6):
    scores = bm25.get_scores(tokenize_for_bm25(query))
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{
        "index": int(index),
        "text": chosen_chunks[index]["text"],
        "score": float(scores[index]),
        "source": chosen_chunks[index]["source"],
        "page": chosen_chunks[index]["page"]
    } for index in top_indices]

print("BM25 index created.")

BM25 index created.


## 6. Hybrid Retrieval using Reciprocal Rank Fusion

Since I have two ways of retrieving chunks now, dense and BM25, I combine their rankings together using reciprocal rank fusion. This way a chunk that ranks well in either search still gets picked up.


In [11]:
RETRIEVAL_K = 6

def hybrid_search(query, top_k=RETRIEVAL_K, dense_weight=0.5, bm25_weight=0.5):
    dense_results = dense_search(query, top_k)
    keyword_results = bm25_search(query, top_k)
    combined_scores = {}

    for rank, result in enumerate(dense_results, 1):
        idx = result["index"]
        combined_scores[idx] = combined_scores.get(idx, 0) + dense_weight / (60 + rank)

    for rank, result in enumerate(keyword_results, 1):
        idx = result["index"]
        combined_scores[idx] = combined_scores.get(idx, 0) + bm25_weight / (60 + rank)

    ranked_indices = sorted(combined_scores, key=combined_scores.get, reverse=True)

    return [{
        "index": idx,
        "text": chosen_chunks[idx]["text"],
        "hybrid_score": combined_scores[idx],
        "source": chosen_chunks[idx]["source"],
        "page": chosen_chunks[idx]["page"]
    } for idx in ranked_indices[:top_k]]

print("Hybrid retrieval initialized.")

Hybrid retrieval initialized.


## 7. Cross-Encoder Re-ranking

After hybrid search gives me a list of candidate chunks, I pass them through a cross-encoder model that looks at the question and each chunk together and gives a better relevance score. I keep only the top few after this step.


In [12]:
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
FINAL_TOP_K = 3
reranker = CrossEncoder(RERANKER_MODEL_NAME)

def rerank_results(query, retrieved_results, top_n=FINAL_TOP_K):
    if not retrieved_results:
        return []

    pairs = [[query, result["text"]] for result in retrieved_results]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(retrieved_results, scores), key=lambda x: x[1], reverse=True)

    final_results = []
    for result, score in ranked[:top_n]:
        result = result.copy()
        result["rerank_score"] = float(score)
        final_results.append(result)

    return final_results

def retrieve(query, use_hybrid=True, use_reranker=True):
    results = hybrid_search(query) if use_hybrid else dense_search(query, RETRIEVAL_K)

    if use_reranker:
        results = rerank_results(query, results)

    return results

print("Final retrieval pipeline initialized.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Final retrieval pipeline initialized.


In [13]:
test_question = "What is the main idea of this document?"
retrieved_results = retrieve(test_question)

print("QUESTION:", test_question)

for i, result in enumerate(retrieved_results, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(result["text"])

QUESTION: What is the main idea of this document?

--- Retrieved Chunk 1 ---
timent analysis to classify article tone as positive, negative, or neutral alongside the prediction. •Designed a credibility scoring system combining model confidence with source reputation into a single per-article trust score. AI Text Summarizer|Python, NLTK, Flask/github •Built anextractive summarizerusingNLTKthat scores sentences by normalized word frequency to surface the most informative content from any input length. •Implemented stopword removal and punctuation stripping for cleaner scoring; exposed a configurable sentencesparameter so output length can be adjusted without touching core logic.

--- Retrieved Chunk 2 ---
Himanshu Singh Gahlot Jaipur ♂phone+91-8890795885 |/envel⌢pehimanshusinghgahlot1312@gmail.com |/linkedin-inLinkedIn |/githubGitHub |/gl⌢bePortfolio SUMMARY Final-year Computer Science student with a focused interest in Data Science and Machine Learning. Have worked with real datasets, b

## 8. Local Language Model

For generating the actual answer, I used Qwen2.5-1.5B-Instruct. It's small enough to run on the Colab GPU without any issues and was good enough for this kind of document.


In [14]:
LOCAL_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
local_llm = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded:", LOCAL_MODEL_NAME)
print("Device:", local_llm.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


In [15]:
def generate_answer(question, context):
    system_prompt = (
        "You are a grounded question answering assistant. "
        "Answer using ONLY the provided context. "
        "If the answer is not present in the context, say exactly: "
        "\"I don't know based on the provided document.\""
    )

    user_prompt = f"""Context:
{context}

Question:
{question}

Answer:"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(local_llm.device)

    with torch.no_grad():
        outputs = local_llm.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer


def full_pipeline(question, confidence_threshold=-10.5):
    start_time = time.time()

    retrieved_chunks = hybrid_search(question, top_k=6)
    reranked_chunks = rerank_results(question, retrieved_chunks)

    if not reranked_chunks:
        return {
            "question": question,
            "answer": "I don't know based on the provided document.",
            "chunks_used": 0,
            "total_latency": time.time() - start_time
        }

    best_score = reranked_chunks[0]["rerank_score"]

    if best_score < confidence_threshold:
        return {
            "question": question,
            "answer": "I don't know based on the provided document.",
            "chunks_used": len(reranked_chunks),
            "total_latency": time.time() - start_time
        }

    context = "\n\n".join(
        item["text"] for item in reranked_chunks
    )

    answer = generate_answer(question, context)

    return {
        "question": question,
        "answer": answer,
        "chunks_used": len(reranked_chunks),
        "total_latency": time.time() - start_time
    }

In [16]:
question = "What is the main idea of this document?"
result = full_pipeline(question)

print("QUESTION:", result["question"])
print("\nANSWER:", result["answer"])
print(f"\nChunks used: {result['chunks_used']}")
print(f"Total latency: {result['total_latency']:.2f}s")

QUESTION: What is the main idea of this document?

ANSWER: The main idea of this document is to describe various projects and skills related to data science and machine learning, focusing on text processing techniques such as sentiment analysis and natural language processing (NLP). The author shares their experience in building and deploying machine learning models, including training an algorithm to detect fake news and integrating APIs for real-time headline updates.

Chunks used: 3
Total latency: 4.01s


In [17]:
out_of_scope_result = full_pipeline("What is the capital of France?")
print(out_of_scope_result["answer"])

I don't know based on the provided document.


## 9. Evaluation Dataset

I wrote down some questions along with what the answer should be, based on my own resume. This way I can actually check if the retrieval part is finding the right information instead of just trusting the output.


In [18]:
eval_set = [
    {"question": "What is Himanshu's CGPA?", "expected_answer": "8.83"},
    {"question": "Which Linux certification did Himanshu clear?", "expected_answer": "RHCSA"},
    {"question": "What score did Himanshu get in the RHCSA exam?", "expected_answer": "285/300"},
    {"question": "What project did Himanshu build using TF-IDF and Logistic Regression?", "expected_answer": "TruthLens"},
    {"question": "How many labeled articles were used to train the fake news model?", "expected_answer": "40,000+"},
    {"question": "Which library was used to build the extractive summarizer?", "expected_answer": "NLTK"},
    {"question": "What Salesforce Trailhead badge did Himanshu achieve?", "expected_answer": "Agentblazer Innovator"},
    {"question": "Which college is Himanshu studying at?", "expected_answer": "Jaipur Engineering College and Research Centre"},
    {"question": "What programming languages does Himanshu know?", "expected_answer": "C++, Python"},
    {"question": "What hackathon did Himanshu participate in?", "expected_answer": "Innovation Hackathon — ICI FEST 2025"}
]

print("Evaluation questions:", len(eval_set))

Evaluation questions: 10


In [19]:
retrieval_results = []

for item in eval_set:
    retrieved = retrieve(item["question"])
    context = " ".join(result["text"] for result in retrieved).lower()
    hit = item["expected_answer"].lower() in context

    retrieval_results.append({
        "question": item["question"],
        "expected": item["expected_answer"],
        "retrieval_hit": "Yes" if hit else "No",
        "chunks_used": len(retrieved)
    })

retrieval_df = pd.DataFrame(retrieval_results)
retrieval_df

,question,expected,retrieval_hit,chunks_used
0,What is Himanshu's CGPA?,8.83,Yes,3
1,Which Linux certification did Himanshu clear?,RHCSA,Yes,3
2,What score did Himanshu get in the RHCSA exam?,285/300,Yes,3
3,What project did Himanshu build using TF-IDF a...,TruthLens,Yes,3
4,How many labeled articles were used to train t...,"40,000+",Yes,3
5,Which library was used to build the extractive...,NLTK,Yes,3
6,What Salesforce Trailhead badge did Himanshu a...,Agentblazer Innovator,Yes,3
7,Which college is Himanshu studying at?,Jaipur Engineering College and Research Centre,Yes,3
8,What programming languages does Himanshu know?,"C++, Python",Yes,3
9,What hackathon did Himanshu participate in?,Innovation Hackathon — ICI FEST 2025,Yes,3


In [20]:
total = len(retrieval_df)
hits = (retrieval_df["retrieval_hit"] == "Yes").sum()
hit_rate = hits / total * 100

print("=" * 55)
print("RAG RETRIEVAL EVALUATION REPORT")
print("=" * 55)
print(f"Total Questions      : {total}")
print(f"Successful Hits      : {hits}")
print(f"Failed Hits          : {total - hits}")
print(f"Retrieval Hit Rate   : {hit_rate:.2f}%")
print(f"Final Chunks Used    : {FINAL_TOP_K}")
print("=" * 55)

RAG RETRIEVAL EVALUATION REPORT
Total Questions      : 10
Successful Hits      : 10
Failed Hits          : 0
Retrieval Hit Rate   : 100.00%
Final Chunks Used    : 3


## 10. Experiment 1: Comparing Chunking Strategies

Going back to the three chunking strategies from earlier, I ran the evaluation questions on each one to see which chunk size actually retrieves the correct information more often.


In [21]:
chunk_experiment_results = []

for strategy_name, strategy_chunks in chunk_results.items():
    texts = [chunk["text"] for chunk in strategy_chunks]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    temp_index = faiss.IndexFlatIP(embeddings.shape[1])
    temp_index.add(embeddings)
    hits = 0

    for item in eval_set:
        q = embedding_model.encode(
            [item["question"]],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        _, indices = temp_index.search(q, 3)
        context = " ".join(texts[i] for i in indices[0] if i != -1).lower()

        if item["expected_answer"].lower() in context:
            hits += 1

    chunk_experiment_results.append({
        "Strategy": strategy_name,
        "Chunks": len(strategy_chunks),
        "Successful Hits": hits,
        "Hit Rate (%)": round(hits / len(eval_set) * 100, 2)
    })

pd.DataFrame(chunk_experiment_results)

,Strategy,Chunks,Successful Hits,Hit Rate (%)
0,small_overlap,18,9,90.0
1,medium_overlap,10,8,80.0
2,large_overlap,6,9,90.0


## 11. Experiment 2: Comparing Embedding Models

I also tried swapping the embedding model to see if a bigger model gives better retrieval or if the smaller one is already good enough for a document this small.


In [22]:
embedding_models_to_test = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "MPNet": "sentence-transformers/all-mpnet-base-v2"
}

embedding_experiment_results = []

for label, model_name in embedding_models_to_test.items():
    print("Testing:", label)
    temp_model = SentenceTransformer(model_name)

    embeddings = temp_model.encode(
        chunk_texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    temp_index = faiss.IndexFlatIP(embeddings.shape[1])
    temp_index.add(embeddings)
    hits = 0

    for item in eval_set:
        q = temp_model.encode(
            [item["question"]],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        _, indices = temp_index.search(q, FINAL_TOP_K)
        context = " ".join(chunk_texts[i] for i in indices[0] if i != -1).lower()

        if item["expected_answer"].lower() in context:
            hits += 1

    embedding_experiment_results.append({
        "Embedding Model": label,
        "Dimensions": embeddings.shape[1],
        "Successful Hits": hits,
        "Hit Rate (%)": round(hits / len(eval_set) * 100, 2)
    })

    del temp_model, embeddings, temp_index
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pd.DataFrame(embedding_experiment_results)

Testing: MiniLM


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Testing: MPNet


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,Embedding Model,Dimensions,Successful Hits,Hit Rate (%)
0,MiniLM,384,9,90.0
1,MPNet,768,8,80.0


## 12. Experiment 3: Dense vs Hybrid vs Hybrid with Re-ranking

Here I compare all three retrieval setups I built, plain dense search, hybrid search, and hybrid with re-ranking added on top, to see how much difference each step actually makes.


In [23]:
configurations = [
    ("Dense Only", False, False),
    ("Hybrid Search", True, False),
    ("Hybrid + Reranking", True, True)
]

rows = []

for name, use_hybrid, use_reranker in configurations:
    hits = 0

    for item in eval_set:
        retrieved = retrieve(item["question"], use_hybrid, use_reranker)
        context = " ".join(r["text"] for r in retrieved).lower()

        if item["expected_answer"].lower() in context:
            hits += 1

    rows.append({
        "Retrieval Method": name,
        "Successful Hits": hits,
        "Total Questions": len(eval_set),
        "Hit Rate (%)": round(hits / len(eval_set) * 100, 2)
    })

retrieval_experiment_df = pd.DataFrame(rows)
retrieval_experiment_df

,Retrieval Method,Successful Hits,Total Questions,Hit Rate (%)
0,Dense Only,10,10,100.0
1,Hybrid Search,10,10,100.0
2,Hybrid + Reranking,10,10,100.0


## 13. Full Generation Evaluation

This runs the complete pipeline, retrieval plus generation, on every question in the evaluation set. It takes a little longer since the model has to generate a full answer each time instead of just retrieving chunks.


In [24]:
generation_results = []

for item in eval_set:
    result = full_pipeline(item["question"])
    correct = item["expected_answer"].lower() in result["answer"].lower()

    generation_results.append({
        "question": item["question"],
        "expected": item["expected_answer"],
        "generated_answer": result["answer"],
        "correct": "Yes" if correct else "No",
        "latency_sec": round(result["total_latency"], 2),
        "chunks_used": result["chunks_used"]
    })

generation_df = pd.DataFrame(generation_results)
generation_df

,question,expected,generated_answer,correct,latency_sec,chunks_used
0,What is Himanshu's CGPA?,8.83,The provided text does not contain information...,No,1.24,3
1,Which Linux certification did Himanshu clear?,RHCSA,The Linux certification that Himanshu cleared ...,Yes,1.24,3
2,What score did Himanshu get in the RHCSA exam?,285/300,He cleared the Red Hat Certified System Admini...,Yes,1.58,3
3,What project did Himanshu build using TF-IDF a...,TruthLens,TruthLens | AI News & Credibility Engine,Yes,0.98,3
4,How many labeled articles were used to train t...,"40,000+",The number of labeled articles used to train t...,Yes,1.45,3
5,Which library was used to build the extractive...,NLTK,The extractive summarizer was built using NLTK...,Yes,1.15,3
6,What Salesforce Trailhead badge did Himanshu a...,Agentblazer Innovator,He achieved the Agentblazer Innovator 2026badg...,Yes,1.48,3
7,Which college is Himanshu studying at?,Jaipur Engineering College and Research Centre,Jaipur Engineering College and Research Centre,Yes,1.02,3
8,What programming languages does Himanshu know?,"C++, Python",C++ and Python,No,0.80,3
9,What hackathon did Himanshu participate in?,Innovation Hackathon — ICI FEST 2025,Himanshu participated in an Innovention Hackat...,No,1.50,3


In [25]:
retrieval_accuracy = (retrieval_df["retrieval_hit"] == "Yes").mean() * 100
generation_accuracy = (generation_df["correct"] == "Yes").mean() * 100
average_latency = generation_df["latency_sec"].mean()

print("=" * 60)
print("FINAL RAG SYSTEM EVALUATION")
print("=" * 60)
print(f"Retrieval Hit Rate      : {retrieval_accuracy:.2f}%")
print(f"Generation Accuracy     : {generation_accuracy:.2f}%")
print(f"Average Response Time   : {average_latency:.2f} seconds")
print(f"Initial Candidates      : {RETRIEVAL_K}")
print(f"Final Context Chunks    : {FINAL_TOP_K}")
print(f"Chunk Strategy          : {CHOSEN_STRATEGY}")
print(f"Embedding Model         : {EMBEDDING_MODEL_NAME}")
print(f"Generation Model        : {LOCAL_MODEL_NAME}")
print("=" * 60)

FINAL RAG SYSTEM EVALUATION
Retrieval Hit Rate      : 100.00%
Generation Accuracy     : 70.00%
Average Response Time   : 1.24 seconds
Initial Candidates      : 6
Final Context Chunks    : 3
Chunk Strategy          : large_overlap
Embedding Model         : sentence-transformers/all-MiniLM-L6-v2
Generation Model        : Qwen/Qwen2.5-1.5B-Instruct


## 14. Simple Notebook UI

Just a small text box where I can type any question about the document and get an answer without having to run a code cell manually every time.


In [26]:
question = input("Ask a question about the uploaded document: ")

if question.strip():
    print("\nSearching document...\n")

    result = full_pipeline(question.strip())

    print("ANSWER")
    print("=" * 50)
    print(result["answer"])
    print()
    print(f"Chunks used: {result['chunks_used']}")
    print(f"Response time: {result['total_latency']:.2f} seconds")
else:
    print("Please enter a question.")

Ask a question about the uploaded document: His skills

Searching document...

ANSWER
Based on the provided context, His skills include:

- Familiarity with algorithms, object-oriented programming, database management systems, operating systems, and platforms such as Linux and VMware.
- Proficiency in Python and its libraries like NumPy, Pandas, NLTK, and ML algorithms.
- Experience with data preprocessing techniques.
- Knowledge of computer fundamentals including data structures, algorithms, and Object-Oriented Programming (OOP).
- Ability to work comfortably with both Python and Linux environments.
- Quick learning ability when it comes to picking up new tools.
- Preference for projects that allow measurable results and iterative improvement.

Chunks used: 3
Response time: 5.79 seconds


## Key Observations

Some things I noticed while working on this:

Small chunks kept things focused but sometimes split related information into two different chunks.
Larger chunks kept context together better, at the cost of carrying some extra text that wasn't needed.
Dense retrieval was better at understanding what the question meant, while BM25 was better at catching exact words like names or numbers.
Combining both together worked better than using either one alone.
Re-ranking helped push the most relevant chunk to the top when the hybrid search returned a few close candidates.
My retrieval hit rate ended up higher than my generation accuracy. I think this is because the model sometimes answers correctly but phrases it differently than the exact expected text, so it still gets marked wrong even though the answer is fine.


## Conclusion

Overall this assignment helped me actually understand how a RAG pipeline works instead of just reading about it. I built the ingestion, chunking, embedding, retrieval and generation steps one by one and tested different options at each step instead of just picking the first thing that worked.

The experiments showed that small design choices, like chunk size or whether to use hybrid search, actually change the results, which was the main thing I was supposed to learn from this assignment.
